This code is to check and fix train, test and validation subset of bee images. At first, we will generate all the species labels for train, validation, and test subsets. Then we will move the images from validation and test to train if that class is not present in the train subset.

In [13]:
import os
from tqdm import tqdm
import pandas as pd
import shutil

In [14]:
# Class names of every image
fulldata_path = r"/home/c/choton/beemachine/datasets/Bee_Full_Body_Segments/Images"
DATA_DIR = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v5/"

# Get list of classes
classes = os.listdir(fulldata_path)
species_dict = {}

for c in classes:
    images_path = os.path.join(fulldata_path, c)
    images = os.listdir(images_path)
    for image in images:
        image_name = image[:-4]
        species_dict[image_name] = c
image_names = list(species_dict.keys())
print(f"Total images for the original data with class labels: {len(image_names)}")

Total images for the original data with class labels: 8029


In [15]:
train_imgs_path = os.path.join(DATA_DIR, 'train', 'aug_images')
val_imgs_path = os.path.join(DATA_DIR, 'valid', 'images')
test_imgs_path = os.path.join(DATA_DIR, 'test', 'images')
train_images = os.listdir(train_imgs_path)
val_images = os.listdir(val_imgs_path)
test_images = os.listdir(test_imgs_path)
print(f"Images in dataset, train: {len(train_images)}, val: {len(val_images)}, test: {len(test_images)}")

Images in dataset, train: 34722, val: 1158, test: 771


In [16]:
# Preprocess image_names ONCE
normalized_species = {
    image_n.replace("._", "-_").replace(".jpg", ""): species_dict[image_n]
    for image_n in image_names
}

Generate labels for train subset

In [17]:
count = 0
species_y = {}
# Loop only once
for image_y in tqdm(train_images):
    for key, species in normalized_species.items():
        if key in image_y:
            count += 1
            species_y[image_y] = species
            break   # IMPORTANT: stop after first match
print(count)

train_dict = {'images': train_images}
train_df = pd.DataFrame(train_dict)
train_df['species'] = train_df['images'].map(species_y)
train_df.head()

  0%|          | 0/34722 [00:00<?, ?it/s]

100%|██████████| 34722/34722 [00:23<00:00, 1479.05it/s]


34722


,images,species
0,Bombus_distinguendus_GBIF_iNat_2573822774_2-jp...,Bombus_distinguendus
1,Bombus_bifarius_47362646_1_1_jpg.rf.bd7d10af8d...,Bombus_bifarius
2,BBW_Bombus_vandykei_4734_549ccb25-aae0-45c5-b0...,Bombus_vandykei
3,20DQ50CQ40DQG000G0DQ20TRQQYRKQTRSQZ0I0Z090TRXQ...,Bombus_fervidus
4,BBW_Bombus_sitkensis_1695_537b7c71-eed0-4246-a...,Bombus_sitkensis


Generate labels for val subset

In [18]:
count = 0
species_y = {}
# Loop only once
for image_y in tqdm(val_images):
    for key, species in normalized_species.items():
        if key in image_y:
            count += 1
            species_y[image_y] = species
            break   # IMPORTANT: stop after first match
print(count)
valid_dict = {'images': val_images}
valid_df = pd.DataFrame(valid_dict)
valid_df['species'] = valid_df['images'].map(species_y)
valid_df.head()

100%|██████████| 1158/1158 [00:00<00:00, 1449.60it/s]

1158


,images,species
0,BBW_Bombus_appositus_23855_e5f978d9-d6e7-4322-...,Bombus_appositus
1,Bombus_lapidarius_iNat_12287097_1-jpg_jpg.rf.3...,Bombus_lapidarius
2,0HGRDZIROZ7RFZXRULHZ9L0R3ZYLNLKROZRZ9LYLNLZZTZ...,Bombus_vosnesenskii
3,Bombus_morio_GBIF_iNat_2603344712_9-jpg_jpg.rf...,Bombus_morio
4,Bombus_pratorum_iNat_21415168_1-jpg_jpg.rf.0db...,Bombus_pratorum


Generate labels for test subset

In [19]:
count = 0
species_y = {}
# Loop only once
for image_y in tqdm(test_images):
    for key, species in normalized_species.items():
        if key in image_y:
            count += 1
            species_y[image_y] = species
            break   # IMPORTANT: stop after first match
print(count)
test_dict = {'images': test_images}
test_df = pd.DataFrame(test_dict)
test_df['species'] = test_df['images'].map(species_y)
test_df.head()

  0%|          | 0/771 [00:00<?, ?it/s]

100%|██████████| 771/771 [00:00<00:00, 1457.75it/s]

771


,images,species
0,Bombus_robustus_GBIF_iNat_2814215056_7-jpg_jpg...,Bombus_robustus
1,5QV06QB0AQ6K8KAK8K1KLKVK7KAKUQLSXK2K5QB0UQ305K...,Bombus_appositus
2,60CQ70CQ6000E0FRQQH0E000IQ3R20FRQQTR20L090YRE0...,Bombus_appositus
3,BBW_Bombus_appositus_28231_846be2d2-b3b3-4947-...,Bombus_appositus
4,Bombus_hortulanus_GBIF_iNat_2028461117_1-jpg_j...,Bombus_hortulanus


In [25]:
n_train = len(set(train_df['species']))
n_val = len(set(valid_df['species']))
n_test = len(set(test_df['species']))
print(f"Total classes in the subsets:")
print("Train:", n_train)
print("Val:", n_val)
print("Test:", n_test)

Total classes in the subsets:
Train: 160
Val: 114
Test: 113


In [21]:
val_train_dif = set(valid_df['species'])-set(train_df['species'])
print("Species that are in valid, but not in train subset:")
val_train_dif

Species that are in valid, but not in train subset:


set()

In [22]:
test_train_dif = set(test_df['species'])-set(train_df['species'])
print("Species that are in test, but not in train subset:")
test_train_dif

Species that are in test, but not in train subset:


set()

In [23]:
train_df_path = os.path.join(DATA_DIR, 'train_aug_labels.csv')
val_df_path = os.path.join(DATA_DIR, 'val_labels.csv')
test_df_path = os.path.join(DATA_DIR, 'test_labels.csv')
train_df.to_csv(train_df_path, index=False)
valid_df.to_csv(val_df_path, index=False)
test_df.to_csv(test_df_path, index=False)

In [24]:
n_train = len(set(train_df['species']))
n_val = len(set(valid_df['species']))
n_test = len(set(test_df['species']))
print(f"Total classes in the subsets:")
print("Train:", n_train)
print("Val:", n_val)
print("Test:", n_test)

Total classes in the subsets:
Train: 160
Val: 114
Test: 113
